# 13 — Measurement Confidence Engine

This notebook demonstrates the **QTc Measurement Confidence Score**,
an explainable composite metric (0–100) that integrates:

1. Signal quality
2. T-end method stability
3. T-wave morphology risk
4. Formula agreement
5. Beat-to-beat stability

**Interpretation tiers:**
- 90–100: High Confidence
- 70–89: Review Recommended
- <70: Manual Review Required


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ecg_analytics.confidence import (
    measurement_confidence,
    signal_quality_subscore,
    formula_agreement_score,
    beat_stability_score,
)
from ecg_analytics.tend.agreement import compute_agreement
from ecg_analytics.confidence.tend_stability import tend_stability_subscore
from ecg_analytics.confidence.morphology_risk import morphology_risk_subscore
from ecg_analytics.morphology import classify_t_wave


## Generate a synthetic ECG signal


In [ ]:
fs = 500.0  # Hz
t = np.linspace(0, 2, int(2 * fs))
# Simple synthetic: R-peak at t=0.5s, T-wave around t=0.75s
signal = (
    1.5 * np.exp(-((t - 0.5) ** 2) / (2 * 0.005**2))   # R-peak
    + 0.4 * np.exp(-((t - 0.75) ** 2) / (2 * 0.03**2))  # T-wave
)
plt.figure(figsize=(12, 3))
plt.plot(t, signal)
plt.title('Synthetic ECG')
plt.xlabel('Time (s)')
plt.show()


## Compute sub-scores


In [ ]:
# Signal quality
sq_score = signal_quality_subscore(signal, fs)
print(f'Signal Quality Sub-score: {sq_score:.1f}')

# T-end stability (run all 4 methods)
t_peak = int(0.75 * fs)
agreement = compute_agreement(signal, t_peak, fs)
ts_score = tend_stability_subscore(agreement)
print(f'T-End Stability Sub-score: {ts_score:.1f}')
print(f'  Method results: {agreement.method_results}')

# Morphology
t_start = int(0.65 * fs)
t_end_idx = int(0.90 * fs)
morph = classify_t_wave(signal, t_start, t_end_idx, fs)
mr_score = morphology_risk_subscore(morph.morphology, morph.confidence)
print(f'Morphology Risk Sub-score: {mr_score:.1f} (type: {morph.morphology})')

# Formula agreement
fa = formula_agreement_score(qt_ms=400.0, rr_ms=850.0)
print(f'Formula Agreement Sub-score: {fa.agreement_score:.1f} (spread: {fa.spread_ms:.1f} ms)')

# Beat stability (simulated beat-level QTs)
beat_qts = [398.0, 402.0, 400.0, 399.5, 401.0, 400.5, 399.0, 401.5, 400.0, 398.5]
bs = beat_stability_score(beat_qts)
print(f'Beat Stability Sub-score: {bs.stability_score:.1f} (CV: {bs.cv:.4f})')


## Composite Confidence Score


In [ ]:
result = measurement_confidence(
    signal_quality=sq_score,
    tend_stability=ts_score,
    morphology=mr_score,
    formula_agreement=fa.agreement_score,
    beat_stability=bs.stability_score,
)
print(f'Measurement Confidence Score: {result.score:.1f}')
print(f'Tier: {result.tier}')
print(f'Sub-scores: {result.sub_scores}')


## Visualization — Sub-score Breakdown


In [ ]:
labels = list(result.sub_scores.keys())
values = list(result.sub_scores.values())

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
ax.barh(labels, values, color=colors)
ax.set_xlim(0, 105)
ax.axvline(90, color='green', linestyle='--', alpha=0.5, label='High threshold')
ax.axvline(70, color='orange', linestyle='--', alpha=0.5, label='Review threshold')
for i, v in enumerate(values):
    ax.text(v + 1, i, f'{v:.1f}', va='center')
ax.set_title(f'Confidence Breakdown (Overall: {result.score:.1f} — {result.tier})')
ax.legend()
plt.tight_layout()
plt.show()
